# akb × GeoAgent — Interactive Demo

This notebook shows how to connect the Archive Knowledge Base (`akb`) to
GeoAgent so you can query the corpus with natural language and render results
on a live Leaflet map — with optional overlay of satellite / STAC data.

## Install
```bash
pip install -e '../[llm,geoagent]'
python -m spacy download en_core_web_sm
```

## Quickstart pipeline (if not already run)
```bash
akb ingest sample_data/lake_chad_climate.md
akb ingest sample_data/water_harvesting_sahel.md
akb ingest sample_data/kanem_bornu_empire.md
akb ingest sample_data/boko_haram_political_economy.md
akb ingest sample_data/decentralised_finance_africa.md
akb chunk --all && akb embed --all && akb ner --all
akb resolve geo --all && akb resolve chrono --all
```

In [1]:
import sys, pathlib
# Make sure akb is importable from the notebook
sys.path.insert(0, str(pathlib.Path('..').resolve()))

## 1 · Direct tool calls (no GeoAgent install required)

Every akb tool returns plain dicts / GeoJSON — you can call them without
the full GeoAgent stack and load results directly into leafmap.

In [19]:
from cli.geoagent_tools import (
    akb_search_locations,
    akb_get_timeline_locations,
    akb_get_entity_network,
    akb_export_geojson,
)

### 1a · Semantic search → map

In [3]:
import leafmap

geojson = akb_search_locations("water harvesting flood mitigation agricultural yield", top_k=12)
print(f"{len(geojson['features'])} location features")

m = leafmap.Map(center=[12, 10], zoom=5)
m.add_geojson(geojson, layer_name="Water harvesting search")
m

35 location features


Map(center=[12, 10], controls=(ZoomControl(options=['position', 'zoom_in_text', 'zoom_in_title', 'zoom_out_tex…

### 1b · Temporal filter — medieval period (900–1500 CE)

In [4]:
medieval = akb_get_timeline_locations(iso_start="0900", iso_end="1500")
print(f"{len(medieval['features'])} features in 900–1500 CE window")
for f in medieval['features'][:6]:
    p = f['properties']
    print(f"  {p['name']:<35} t={p['time_context']:<20} [{p['block_title']}]")

17 features in 900–1500 CE window
  Lake Chad                           t=0901–1000            [Lake Chad Climate]
  Borno State, Nigeria                t=1301–1400            [Lake Chad Climate]
  Diffa, Niger                        t=1301–1400            [Lake Chad Climate]
  Kukawa, Borno, Nigeria              t=1301–1400            [Kanem Bornu Empire]
  Kukawa, Borno, Nigeria              t=1301–1400            [Kanem Bornu Empire]
  Lake Chad                           t=1301–1400            [Lake Chad Climate]


In [5]:
m2 = leafmap.Map(center=[12, 13], zoom=5)
m2.add_geojson(medieval, layer_name="Medieval period (900–1500 CE)")
m2

Map(center=[12, 13], controls=(ZoomControl(options=['position', 'zoom_in_text', 'zoom_in_title', 'zoom_out_tex…

### 1c · Entity network — what does the corpus say about Maiduguri?

In [6]:
import json
network = akb_get_entity_network("Maiduguri")
print(f"{network['occurrence_count']} chunks mention Maiduguri\n")
for occ in network['occurrences']:
    print(f"[{occ['block_title']}]")
    print(f"  excerpt: {occ['excerpt'][:120]}...")
    for t, vals in occ['co_entities'].items():
        print(f"  {t}: {', '.join(vals[:4])}")
    print()

7 chunks mention Maiduguri

[Lake Chad Climate]
  excerpt: ## Link to Conflict and Displacement  Research published between 2015 and 2022 has increasingly drawn connections betwee...
  TIME: 2015–2015, 2017-04-01–2017-04-01, 2000–2000, 2015-04-01–2015-04-01
  PERSON: Yobe, Boko Haram's, N'Djamena
  ORG: the United Nations Environment Programme, The Multinational Joint Task Force

[Decentralised Finance Africa]
  excerpt: ## Structural Limitations and Failure Modes  ROSCAs function on trust and social sanction. Their limitations in contexts...
  ORG: Esther Duflo's, MIT
  TIME: 2010–2010

[Boko Haram Political Economy]
  excerpt: ## Overview  Boko Haram — formally Jamā'at Ahl as-Sunnah lid-Da'wah wa'l-Jihād — emerged in Maiduguri, the capital of Bo...
  PERSON: Jamā'at Ahl, Mohammed Yusuf, Yobe, Boko Haram
  TIME: 2002-04-01–2002-04-01, 2014-04-01–2014-04-01, 2014–2014
  ORG: Nigerian

[Boko Haram Political Economy]
  excerpt: ## The Economic Displacement Thesis  A dominant early narrat

## 2 · Full GeoAgent session

Requires `pip install geoagent leafmap`. The agent is created via `for_leafmap`
so it has the full suite of built-in map-control tools (add/remove/list layers,
zoom, basemap, STAC …) plus the four akb corpus tools as extras.

`agent.chat(query)` returns a `GeoAgentResponse`. Use `show(resp)` to
pretty-print the answer as Markdown.

In [14]:
!python3 -m venv .venv-ipynb && source .venv-ipynb/bin/activate && python3 -m pip install geoagent leafmap

  Using cached geoagent-1.1.1-py2.py3-none-any.whl.metadata (16 kB)
  Using cached leafmap-0.62.0-py2.py3-none-any.whl.metadata (17 kB)
  Using cached strands_agents-1.37.0-py3-none-any.whl.metadata (18 kB)
  Using cached pydantic-2.13.3-py3-none-any.whl.metadata (108 kB)
  Using cached anywidget-0.11.0-py3-none-any.whl.metadata (8.9 kB)
  Using cached bqplot-0.12.46-py2.py3-none-any.whl.metadata (6.4 kB)
  Using cached duckdb-1.5.2-cp313-cp313-macosx_10_13_x86_64.whl.metadata (4.2 kB)
  Using cached folium-0.20.0-py2.py3-none-any.whl.metadata (4.2 kB)
  Using cached gdown-6.0.0-py3-none-any.whl.metadata (7.4 kB)
  Using cached geojson-3.2.0-py3-none-any.whl.metadata (16 kB)
  Using cached geopandas-1.1.3-py3-none-any.whl.metadata (2.3 kB)
  Using cached ipyevents-2.0.4-py3-none-any.whl.metadata (3.0 kB)
  Using cached ipyfilechooser-0.6.0-py3-none-any.whl.metadata (6.4 kB)
  Using cached ipyleaflet-0.20.0-py3-none-any.whl.metadata (5.3 kB)
  Using cached ipyvue-1.12.0-py2.py3-none-any

In [ ]:
import importlib, cli.geoagent_tools
importlib.reload(cli.geoagent_tools)
from cli.geoagent_tools import make_agent
from IPython.display import display, Markdown

def show(resp):
    """Pretty-print a GeoAgentResponse as Markdown."""
    if not resp.success:
        display(Markdown(f"**Error:** `{resp.error_message}`"))
        return
    lines = []
    if resp.answer_text:
        lines.append(resp.answer_text)
    if resp.executed_tools:
        tools_str = ", ".join(f"`{t}`" for t in resp.executed_tools)
        lines.append(f"\n---\n_Tools used: {tools_str} · {resp.execution_time:.1f}s_")
    display(Markdown("\n".join(lines) if lines else "_No response text._"))

m3 = leafmap.Map(center=[12, 10], zoom=5)
agent = make_agent(map=m3, model="claude-sonnet-4-6")
m3  # display map — agent will add layers to it as it runs

In [29]:
! export $(cat ~/code/surulere/archive-knowledge-base/.env | xargs)

1744.47s - pydevd: Sending message related to process being replaced timed-out after 5 seconds


In [43]:
import os
# print(os.environ.get("ANTHROPIC_API_KEY", "NOT SET"))

In [ ]:
# Multi-step query: search corpus + add locations to map
resp = agent.chat(
    "Search the knowledge base for locations related to climate displacement and "
    "conflict in northeast Nigeria and add them to the map."
)
show(resp)

In [ ]:
# Temporal deep-dive: Kanem-Bornu period
resp = agent.chat(
    "Show all locations from the knowledge base that date to between 900 and 1500 CE, "
    "then tell me which of those locations are within 200 km of Lake Chad."
)
show(resp)

In [ ]:
# Cross-disciplinary synthesis
resp = agent.chat(
    "Use the knowledge base to find everything the archive says about water management "
    "in Borno State across all time periods. Summarise the argument chain from "
    "historical Kanem-Bornu canal systems to modern swale rehabilitation proposals, "
    "citing specific sources and dates."
)
show(resp)

## 3 · Export for external GIS tools

All GeoJSON outputs drop straight into QGIS, Google Earth, or Observable.

In [ ]:
import json, pathlib

# Full corpus GeoJSON → QGIS / Google Earth
full = akb_export_geojson()
pathlib.Path("../data/corpus_locations.geojson").write_text(json.dumps(full, indent=2))
print(f"Wrote {len(full['features'])} features to data/corpus_locations.geojson")

# Filtered: Lake Chad documents only
lake_chad = akb_export_geojson(block_title_filter="lake chad")
pathlib.Path("../data/lake_chad_locations.geojson").write_text(json.dumps(lake_chad, indent=2))
print(f"Wrote {len(lake_chad['features'])} features to data/lake_chad_locations.geojson")

## CLI equivalents

Everything above is also runnable from the terminal:

```bash
# Search → GeoJSON stdout
akb geoagent --tool search-locations --query "water harvesting flood"

# Timeline filter → file
akb geoagent --tool timeline-locations --start 0900 --end 1500 --out medieval.geojson

# Entity network
akb geoagent --tool entity-network --query "Maiduguri"

# Full export
akb geoagent --tool export-geojson --out corpus.geojson

# Full agent prompt
akb geoagent "Show water management sites from 900–1500 CE and overlay MODIS water"
```